In [ ]:
%%sql -r DataImport
-- 1. Create your workspace
CREATE DATABASE IF NOT EXISTS HACKATHON_DB;
CREATE SCHEMA IF NOT EXISTS HACKATHON_DB.ML_CHURN;
USE SCHEMA HACKATHON_DB.ML_CHURN;

-- 2. Create a file format for your CSVs
CREATE OR REPLACE FILE FORMAT csv_format
  TYPE = 'CSV'
  FIELD_OPTIONALLY_ENCLOSED_BY = '"'
  SKIP_HEADER = 1
  NULL_IF = ('', 'NULL')
  FIELD_DELIMITER = ','
  ENCODING = 'UTF8';

-- 3. Create an internal stage
CREATE OR REPLACE STAGE churn_stage
  FILE_FORMAT = csv_format;

-- 4. Create the 5 target tables
CREATE OR REPLACE TABLE RAW_HR_ATTRITION (
  employee_id VARCHAR,
  employee_name VARCHAR,
  department VARCHAR,
  manager_id VARCHAR,
  tenure_years FLOAT,
  performance_rating INT,
  salary_band VARCHAR,
  months_since_promotion INT,
  engagement_score FLOAT,
  team_size INT,
  hire_date DATE,
  attrition_risk_label VARCHAR
);

CREATE OR REPLACE TABLE RAW_ACCESS_LOGS (
  user_id VARCHAR,
  role VARCHAR,
  table_name VARCHAR,
  table_owner VARCHAR,
  last_accessed_date DATE,
  query_count_30d INT
);

CREATE OR REPLACE TABLE RAW_TOOL_ADOPTION (
  user_id VARCHAR,
  team VARCHAR,
  org_unit VARCHAR,
  event_date DATE,
  session_count INT,
  actions_taken INT,
  first_login_date DATE
);

CREATE OR REPLACE TABLE RAW_PROMPT_QUALITY (
  user_id VARCHAR,
  team VARCHAR,
  org_unit VARCHAR,
  log_date DATE,
  prompt_count INT,
  avg_quality_score FLOAT,
  category VARCHAR
);

CREATE OR REPLACE TABLE RAW_CONTRACT_METADATA (
  contract_id VARCHAR,
  vendor_name VARCHAR,
  category VARCHAR,
  contract_value_usd FLOAT,
  start_date DATE,
  end_date DATE,
  governing_law VARCHAR,
  liability_cap_usd FLOAT,
  auto_renews VARCHAR,
  risk_flags VARCHAR,
  risk_level VARCHAR,
  owner_team VARCHAR
);

In [ ]:
%%sql -r Copy_into_tables
COPY INTO RAW_HR_ATTRITION FROM @churn_stage/hr_attrition.csv;
COPY INTO RAW_ACCESS_LOGS FROM @churn_stage/access_logs.csv;
COPY INTO RAW_TOOL_ADOPTION FROM @churn_stage/tool_adoption.csv;
COPY INTO RAW_PROMPT_QUALITY FROM @churn_stage/prompt_quality.csv;
COPY INTO RAW_CONTRACT_METADATA FROM @churn_stage/contract_metadata.csv;

-- Verify
SELECT 'HR' as tbl, COUNT(*) as row_count FROM RAW_HR_ATTRITION
UNION ALL SELECT 'ACCESS', COUNT(*) FROM RAW_ACCESS_LOGS
UNION ALL SELECT 'TOOL', COUNT(*) FROM RAW_TOOL_ADOPTION
UNION ALL SELECT 'PROMPT', COUNT(*) FROM RAW_PROMPT_QUALITY
UNION ALL SELECT 'CONTRACT', COUNT(*) FROM RAW_CONTRACT_METADATA;

In [ ]:
%%sql -r Create_Churn_Label
-- Build churn proxy from MULTIPLE behavioral signals
CREATE OR REPLACE TABLE LABELED_CHURN AS
WITH tool_activity AS (
    SELECT
        CAST(REGEXP_REPLACE(user_id, '[^0-9]', '') AS INT) AS emp_num,
        MAX(event_date) AS last_tool_date,
        AVG(session_count) AS avg_sessions,
        AVG(actions_taken) AS avg_actions,
        COUNT(*) AS total_tool_days
    FROM RAW_TOOL_ADOPTION
    GROUP BY 1
),
access_activity AS (
    SELECT
        CAST(REGEXP_REPLACE(user_id, '[^0-9]', '') AS INT) AS emp_num,
        MAX(last_accessed_date) AS last_access_date,
        AVG(query_count_30d) AS avg_queries
    FROM RAW_ACCESS_LOGS
    GROUP BY 1
),
prompt_activity AS (
    SELECT
        CAST(REGEXP_REPLACE(user_id, '[^0-9]', '') AS INT) AS emp_num,
        MAX(log_date) AS last_prompt_date,
        AVG(prompt_count) AS avg_prompts,
        AVG(avg_quality_score) AS avg_quality
    FROM RAW_PROMPT_QUALITY
    GROUP BY 1
)
SELECT
    h.*,
    t.last_tool_date,
    t.avg_sessions,
    t.avg_actions,
    t.total_tool_days,
    a.last_access_date,
    a.avg_queries,
    p.last_prompt_date,
    p.avg_prompts,
    p.avg_quality,
    -- Churn proxy: combines HR risk + behavioral inactivity
    CASE
        WHEN h.attrition_risk_label = 'High' THEN 1
        WHEN h.engagement_score < 2.5
             AND (t.last_tool_date IS NULL OR DATEDIFF('day', t.last_tool_date, '2026-04-01') > 60) THEN 1
        WHEN t.avg_sessions < 2 AND h.attrition_risk_label = 'Medium' THEN 1
        ELSE 0
    END AS churn_label
FROM RAW_HR_ATTRITION h
LEFT JOIN tool_activity t
    ON CAST(REGEXP_REPLACE(h.employee_id, '[^0-9]', '') AS INT) = t.emp_num
LEFT JOIN access_activity a
    ON CAST(REGEXP_REPLACE(h.employee_id, '[^0-9]', '') AS INT) = a.emp_num
LEFT JOIN prompt_activity p
    ON CAST(REGEXP_REPLACE(h.employee_id, '[^0-9]', '') AS INT) = p.emp_num;

-- Check new churn distribution
SELECT churn_label, COUNT(*) as cnt,
       ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) as pct
FROM LABELED_CHURN
GROUP BY churn_label;

In [ ]:
# Cell 1 - Python: Feature Engineering
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F
from snowflake.snowpark.types import IntegerType

session = get_active_session()

# Load labeled churn table
df = session.table("LABELED_CHURN")

# --- Contract risk features (department-level aggregation) ---
contracts = session.table("RAW_CONTRACT_METADATA")

dept_contract_risk = contracts.group_by(F.col("OWNER_TEAM")).agg(
    F.count("CONTRACT_ID").alias("DEPT_CONTRACT_COUNT"),
    F.avg("CONTRACT_VALUE_USD").alias("DEPT_AVG_CONTRACT_VALUE"),
    F.sum(F.when(F.col("RISK_LEVEL") == 'High', 1).otherwise(0)).alias("DEPT_HIGH_RISK_CONTRACTS"),
    F.avg("LIABILITY_CAP_USD").alias("DEPT_AVG_LIABILITY")
)

# Join contract features to employees via department = owner_team
df = df.join(dept_contract_risk,
             df["DEPARTMENT"] == dept_contract_risk["OWNER_TEAM"],
             join_type="left").drop("OWNER_TEAM")

# --- Encode salary_band to numeric ---
df = df.with_column("SALARY_BAND_NUM",
    F.when(F.col("SALARY_BAND") == 'L1', 1)
     .when(F.col("SALARY_BAND") == 'L2', 2)
     .when(F.col("SALARY_BAND") == 'L3', 3)
     .when(F.col("SALARY_BAND") == 'L4', 4)
     .when(F.col("SALARY_BAND") == 'L5', 5)
)

# --- Derived features ---
df = df.with_column("PROMOTION_STAGNATION",
    F.col("MONTHS_SINCE_PROMOTION") / F.greatest(F.col("TENURE_YEARS"), F.lit(0.5))
)

df = df.with_column("ENGAGEMENT_PER_TEAM",
    F.col("ENGAGEMENT_SCORE") / F.greatest(F.col("TEAM_SIZE"), F.lit(1))
)

# --- Fill nulls for employees without activity data ---
fill_cols = {
    "AVG_SESSIONS": 0.0,
    "AVG_ACTIONS": 0.0,
    "TOTAL_TOOL_DAYS": 0,
    "AVG_QUERIES": 0.0,
    "AVG_PROMPTS": 0.0,
    "AVG_QUALITY": 0.0,
    "DEPT_CONTRACT_COUNT": 0,
    "DEPT_AVG_CONTRACT_VALUE": 0.0,
    "DEPT_HIGH_RISK_CONTRACTS": 0,
    "DEPT_AVG_LIABILITY": 0.0
}

for col_name, fill_val in fill_cols.items():
    df = df.with_column(col_name, F.coalesce(F.col(col_name), F.lit(fill_val)))

# --- Select final feature set (drop non-model columns) ---
feature_table = df.select(
    "EMPLOYEE_ID",
    "TENURE_YEARS",
    "PERFORMANCE_RATING",
    "SALARY_BAND_NUM",
    "MONTHS_SINCE_PROMOTION",
    "ENGAGEMENT_SCORE",
    "TEAM_SIZE",
    "AVG_SESSIONS",
    "AVG_ACTIONS",
    "TOTAL_TOOL_DAYS",
    "AVG_QUERIES",
    "AVG_PROMPTS",
    "AVG_QUALITY",
    "DEPT_CONTRACT_COUNT",
    "DEPT_AVG_CONTRACT_VALUE",
    "DEPT_HIGH_RISK_CONTRACTS",
    "DEPT_AVG_LIABILITY",
    "PROMOTION_STAGNATION",
    "ENGAGEMENT_PER_TEAM",
    "CHURN_LABEL"
)

# Save as training table
feature_table.write.mode("overwrite").save_as_table("FEATURES_TRAINING")

# Verify
session.table("FEATURES_TRAINING").show(5)
print(f"Total rows: {session.table('FEATURES_TRAINING').count()}")
print(f"Total features: {len(session.table('FEATURES_TRAINING').columns) - 2}")  # minus ID and label

In [ ]:
from snowflake.ml._internal import env
print(snowflake.ml.__version__)

In [ ]:
from snowflake.ml.feature_store import FeatureStore, FeatureView, Entity, CreationMode

fs = FeatureStore(
    session=session,
    database="HACKATHON_DB",
    name="ML_CHURN",
    default_warehouse="COMPUTE_WH",
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST
)

entity = Entity(name="SD_EMPLOYEE_ENTITY", join_keys=["EMPLOYEE_ID"])
fs.register_entity(entity)

fv = FeatureView(
    name="SD_CHURN_FEATURE_VIEW",
    entities=[entity],
    feature_df=feature_df,
    desc="Employee churn features: HR + behavioral + contract signals"
)

registered_fv = fs.register_feature_view(feature_view=fv, version="V1")
print(f"Registered: {registered_fv.name} version {registered_fv.version}")

In [ ]:
%%sql -r dataframe_1
SELECT SYSTEM$REFERENCE('VIEW', 'FEATURES_TRAINING_V');

In [ ]:
from snowflake.ml.modeling.classification import RandomForestClassifier
from snowflake.ml.modeling.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
)
from snowflake.snowpark import functions as F

# Load features
train_df = session.table("FEATURES_TRAINING")

# Define feature columns (everything except ID and label)
feature_cols = [c for c in train_df.columns if c not in ["EMPLOYEE_ID", "CHURN_LABEL"]]

# Train/test split
train, test = train_df.random_split([0.8, 0.2], seed=42)

print(f"Train: {train.count()} rows, Test: {test.count()} rows")

# Train classifier
clf = RandomForestClassifier(
    input_cols=feature_cols,
    label_cols=["CHURN_LABEL"],
    output_cols=["PREDICTED_CHURN"]
)

clf.fit(train)

# Predict on test
predictions = clf.predict(test)

# Convert to pandas for metrics
pred_pd = predictions.to_pandas()

auc = roc_auc_score(
    y_true=pred_pd["CHURN_LABEL"],
    y_score=pred_pd["PREDICTED_CHURN"]
)
precision = precision_score(
    y_true=pred_pd["CHURN_LABEL"],
    y_pred=pred_pd["PREDICTED_CHURN"]
)
recall = recall_score(
    y_true=pred_pd["CHURN_LABEL"],
    y_pred=pred_pd["PREDICTED_CHURN"]
)
f1 = f1_score(
    y_true=pred_pd["CHURN_LABEL"],
    y_pred=pred_pd["PREDICTED_CHURN"]
)

print(f"\n--- Model Metrics ---")
print(f"AUC:       {auc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

# Save metrics table for dashboard
session.create_dataframe([{
    "MODEL_NAME": "SD_CHURN_CLASSIFIER",
    "AUC": float(auc),
    "PRECISION": float(precision),
    "RECALL": float(recall),
    "F1_SCORE": float(f1),
    "TRAIN_ROWS": train.count(),
    "TEST_ROWS": test.count()
}]).write.mode("overwrite").save_as_table("MODEL_METRICS")

print("\nMetrics saved to MODEL_METRICS table")

In [ ]:
%%sql -r Model_Training
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE COMPUTE_WH;
USE DATABASE HACKATHON_DB;
USE SCHEMA ML_CHURN;

In [ ]:
%%sql -r dataframe_2
CREATE OR REPLACE VIEW FEATURES_TRAINING_V AS
SELECT
  * EXCLUDE (EMPLOYEE_ID, CHURN_LABEL),
  CHURN_LABEL   -- keep as NUMBER(1,0) or BOOLEAN; both are valid targets
FROM FEATURES_TRAINING;

In [ ]:
import importlib
for pkg in ['xgboost', 'lightgbm', 'sklearn', 'snowflake.ml.modeling', 'snowflake.ml.modeling.ensemble', 'snowflake.ml.modeling.xgboost']:
    try:
        mod = importlib.import_module(pkg)
        print(f"{pkg}: AVAILABLE")
        if 'snowflake' in pkg:
            print(f"  contents: {dir(mod)}")
    except Exception as e:
        print(f"{pkg}: NOT available - {e}")

In [ ]:
# Check snowflake.ml version
import snowflake.ml
print(f"snowflake-ml version: {snowflake.ml.version.VERSION}")

# Check what's inside modeling
import pkgutil, snowflake.ml.modeling
submodules = [name for importer, name, ispkg in pkgutil.walk_packages(snowflake.ml.modeling.__path__, prefix='snowflake.ml.modeling.')]
print("\nAvailable modeling submodules:")
for s in sorted(submodules):
    print(f"  {s}")

In [ ]:
from snowflake.ml.modeling.ensemble import GradientBoostingClassifier
from snowflake.ml.modeling.metrics import precision_score, recall_score, f1_score, roc_auc_score

session = get_active_session()
df = session.table("FEATURES_TRAINING")

feature_cols = [c for c in df.columns if c not in ["EMPLOYEE_ID", "CHURN_LABEL"]]

train_df, test_df = df.random_split([0.8, 0.2], seed=42)
print(f"Train: {train_df.count()}, Test: {test_df.count()}")

clf = GradientBoostingClassifier(
    input_cols=feature_cols,
    label_cols=["CHURN_LABEL"],
    output_cols=["PREDICTED_CHURN"],
    n_estimators=100,
    max_depth=4,
    random_state=42
)
clf.fit(train_df)

predictions = clf.predict(test_df)
pred_pd = predictions.to_pandas()

auc = roc_auc_score(y_true=pred_pd["CHURN_LABEL"], y_score=pred_pd["PREDICTED_CHURN"])
prec = precision_score(y_true=pred_pd["CHURN_LABEL"], y_pred=pred_pd["PREDICTED_CHURN"])
rec = recall_score(y_true=pred_pd["CHURN_LABEL"], y_pred=pred_pd["PREDICTED_CHURN"])
f1 = f1_score(y_true=pred_pd["CHURN_LABEL"], y_pred=pred_pd["PREDICTED_CHURN"])

print(f"\n--- Model Metrics ---")
print(f"AUC:       {auc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1 Score:  {f1:.4f}")

session.create_dataframe([{
    "MODEL_NAME": "SD_CHURN_CLASSIFIER",
    "AUC": float(auc),
    "PRECISION_SCORE": float(prec),
    "RECALL": float(rec),
    "F1_SCORE": float(f1)
}]).write.mode("overwrite").save_as_table("MODEL_METRICS")
print("Metrics saved to MODEL_METRICS table")

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

auc = roc_auc_score(pred_pd["CHURN_LABEL"], pred_pd["PREDICTED_CHURN"])
prec = precision_score(pred_pd["CHURN_LABEL"], pred_pd["PREDICTED_CHURN"])
rec = recall_score(pred_pd["CHURN_LABEL"], pred_pd["PREDICTED_CHURN"])
f1 = f1_score(pred_pd["CHURN_LABEL"], pred_pd["PREDICTED_CHURN"])

print(f"--- Model Metrics ---")
print(f"AUC:       {auc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1 Score:  {f1:.4f}")

session.create_dataframe([{
    "MODEL_NAME": "SD_CHURN_CLASSIFIER",
    "AUC": float(auc),
    "PRECISION_SCORE": float(prec),
    "RECALL": float(rec),
    "F1_SCORE": float(f1)
}]).write.mode("overwrite").save_as_table("MODEL_METRICS")
print("Metrics saved to MODEL_METRICS table")

In [ ]:
from snowflake.ml.registry import Registry

reg = Registry(session=session, database_name="HACKATHON_DB", schema_name="ML_CHURN")

mv = reg.log_model(
    model=clf,
    model_name="SD_CHURN_CLASSIFIER",
    version_name="V3"
)

print(f"Model registered: {mv.model_name} version {mv.version_name}")

In [ ]:
import shap
import pandas as pd

# Extract the underlying sklearn model
model = clf._sklearn_object

# Get test data as pandas
X_test = pred_pd[feature_cols]

# SHAP TreeExplainer
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# Global feature importance
shap_importance = pd.DataFrame({
    "FEATURE_NAME": feature_cols,
    "MEAN_ABS_SHAP": abs(shap_values).mean(axis=0)
}).sort_values("MEAN_ABS_SHAP", ascending=False)

print("--- Top 10 Churn Drivers ---")
print(shap_importance.head(10).to_string(index=False))

# Save to Snowflake
session.create_dataframe(shap_importance).write.mode("overwrite").save_as_table("SHAP_FEATURE_IMPORTANCE")

# Row-level SHAP for Streamlit
shap_df = pd.DataFrame(shap_values, columns=feature_cols)
shap_df["EMPLOYEE_ID"] = pred_pd["EMPLOYEE_ID"].values
shap_df["BASE_VALUE"] = float(explainer.expected_value)
session.create_dataframe(shap_df).write.mode("overwrite").save_as_table("SHAP_VALUES")

print(f"\nSaved: SHAP_FEATURE_IMPORTANCE + SHAP_VALUES ({len(shap_df)} rows)")

In [ ]:
# Score all employees
all_predictions = clf.predict(df)
all_pred_pd = all_predictions.to_pandas()

# Add department for segment breakdowns
labeled = session.table("LABELED_CHURN").select("EMPLOYEE_ID", "DEPARTMENT", "ATTRITION_RISK_LABEL").to_pandas()
scored = all_pred_pd.merge(labeled, on="EMPLOYEE_ID", how="left")

# Save to Snowflake
session.create_dataframe(scored).write.mode("overwrite").save_as_table("PREDICTIONS_BATCH")

print(f"Predictions saved: {len(scored)} rows")
print(f"Predicted churners: {scored['PREDICTED_CHURN'].sum()}")
print(f"\nChurn by department:")
print(scored.groupby("DEPARTMENT")["PREDICTED_CHURN"].mean().sort_values(ascending=False).to_string())